<a href="https://colab.research.google.com/github/DeemonDuck/upi-sentinel/blob/main/inference_engine_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **UPI Fraud Detection - Inference Engine**

##Imports + Path

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import joblib
import json

print("Libraries imported successfully.")

Libraries imported successfully.


## Define Model Artifact Path

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/UPI_SENTINEL/COLAB_NOTEBOOKS/Colab_File_Saved_Checkpoints/"

print("Artifact path set successfully.")

Artifact path set successfully.


## Load Preprocessing Artifacts

In [ ]:
# Load scaler
scaler = joblib.load(SAVE_PATH + "scaler.save")

# Load label encoder
label_encoder = joblib.load(SAVE_PATH + "label_encoder.save")

# Load metadata
with open(SAVE_PATH + "metadata.json", "r") as f:
    metadata = json.load(f)

SEQUENCE_LENGTH = metadata["sequence_length"]
THRESHOLD = metadata["threshold"]

print("Preprocessing artifacts loaded.")
print("Sequence Length:", SEQUENCE_LENGTH)
print("Fraud Threshold:", THRESHOLD)

Preprocessing artifacts loaded.
Sequence Length: 5
Fraud Threshold: 0.99532


## Define Custom Attention Layer

In [ ]:
import tensorflow.keras.backend as K
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        feature_dim = input_shape[-1]

        self.W = self.add_weight(
            name="W",
            shape=(feature_dim, feature_dim),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            name="b",
            shape=(feature_dim,),
            initializer="zeros",
            trainable=True
        )

        self.u = self.add_weight(
            name="u",
            shape=(feature_dim, 1),
            initializer="glorot_uniform",
            trainable=True
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch_size, timesteps, features)

        uit = K.tanh(K.dot(inputs, self.W) + self.b)
        ait = K.dot(uit, self.u)
        ait = K.squeeze(ait, axis=-1)
        a = K.softmax(ait)

        a_expanded = K.expand_dims(a, axis=-1)
        weighted_inputs = inputs * a_expanded

        output = K.sum(weighted_inputs, axis=1)

        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[2])

##Load Trained Model

In [ ]:
model = tf.keras.models.load_model(
    SAVE_PATH + "best_model.keras",
    custom_objects={"AttentionLayer": AttentionLayer}
)

print("Model loaded successfully")

Model loaded successfully


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 32 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## Load Processed Sequences (Demo Input)

In [ ]:
X_sequences = np.load(SAVE_PATH + "X_paysim_seq5_stride1.npy")

print("Dataset Shape:", X_sequences.shape)
print("Single Sequence Shape:", X_sequences[0].shape)

Dataset Shape: (6362616, 5, 6)
Single Sequence Shape: (5, 6)


## Inspect One Example Sequence (Professional Demo Step)

In [ ]:
feature_columns = [
    'type',
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest'
]

example_sequence = X_sequences[0]
display(pd.DataFrame(example_sequence, columns=feature_columns))

,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
0,3.0,-0.291435,-0.265224,-0.270563,-0.323814,-0.333411
1,2.0,-0.296090,-0.288087,-0.292185,-0.320775,-0.333411
2,2.0,-0.282449,-0.284804,-0.291759,-0.315038,-0.328813
3,3.0,-0.295580,-0.283920,-0.288173,-0.323814,-0.333411
4,4.0,0.058703,-0.288472,-0.292442,-0.317217,-0.333411


##Define Inference Function

In [ ]:
def predict_sequence(processed_sequence):

    processed_sequence = np.array(processed_sequence)

    # Validate sequence length only
    if processed_sequence.shape[0] != SEQUENCE_LENGTH:
        raise ValueError(
            f"Expected {SEQUENCE_LENGTH} timesteps, "
            f"but got {processed_sequence.shape[0]}"
        )

    # Dynamic feature count
    processed_sequence = processed_sequence.reshape(
        1, SEQUENCE_LENGTH, processed_sequence.shape[1]
    )

    probability = model.predict(processed_sequence, verbose=0)[0][0]
    decision = "FRAUD" if probability > THRESHOLD else "SAFE"

    return {
        "fraud_probability": float(probability),
        "decision": decision
    }

## Run Fraud Prediction

In [ ]:
result = predict_sequence(example_sequence)
print(result)

{'fraud_probability': 0.45975837111473083, 'decision': 'SAFE'}
